# VibeScent — Multimodal Retrieval (Qwen3-VL-Embedding-8B)

**Branch:** `Text_Processing` | **Model:** `Qwen/Qwen3-VL-Embedding-8B` (MMEB-V2 #1, 77.8)

This notebook pre-computes fragrance embeddings and runs occasion-text + outfit-image retrieval queries against a 10-fragrance test set. Outputs are saved as `.npy` and `.json` artifacts for integration into the Week 3 pipeline.

**Runtime required:** A100 GPU (~16 GB VRAM)
In Colab: `Runtime → Change runtime type → A100 GPU`

---

### What this covers
1. Install GPU deps + verify CUDA
2. Download the Qwen3-VL embedding helper
3. Load `Qwen3-VL-Embedding-8B` in float16
4. Embed 10 test fragrances as documents
5. **Text-only queries** — 4 occasion descriptions
6. **Multimodal queries** — same occasions + outfit image
7. Save artifacts for the repo


## Step 1 — Install dependencies & check GPU

In [ ]:
!pip install -q \
    'transformers>=4.57.0' \
    'qwen-vl-utils>=0.0.14' \
    accelerate \
    huggingface_hub \
    pillow

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
    if gpu.total_memory < 14e9:
        print('WARNING: < 14 GB VRAM detected. Switch to A100 in Runtime settings.')
else:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → GPU (A100 recommended).')


## Step 2 — Download the Qwen3-VL embedding helper

The `qwen3_vl_embedding.py` script is vendored from the model's HuggingFace repo.
It defines `Qwen3VLEmbedder`, a thin wrapper that handles image + text preprocessing and pooling.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

print('Downloading qwen3_vl_embedding.py...')
cached = hf_hub_download(
    repo_id='Qwen/Qwen3-VL-Embedding-8B',
    filename='scripts/qwen3_vl_embedding.py',
)
shutil.copy(cached, './qwen3_vl_embedding.py')
print('OK —', os.path.getsize('./qwen3_vl_embedding.py'), 'bytes')


## Step 3 — Load Qwen3-VL-Embedding-8B

First run downloads ~16 GB of model weights from HuggingFace (cached on subsequent runs).
Load time on A100: ~3-5 minutes first run, ~30 seconds if cached.

In [ ]:
import torch
from qwen3_vl_embedding import Qwen3VLEmbedder

MODEL_ID = 'Qwen/Qwen3-VL-Embedding-8B'
print(f'Loading {MODEL_ID} in float16...')

embedder = Qwen3VLEmbedder(
    model_name_or_path=MODEL_ID,
    torch_dtype=torch.float16,
)
device = next(embedder.model.parameters()).device
print(f'Model loaded on {device}')

# Quick sanity check
with torch.no_grad():
    test = embedder.process([{'text': 'sanity check'}])
print(f'Embedding dim: {test.shape[-1]}')  # 4096 by default


## Step 4 — Load the 10-fragrance test set

Upload `vibescent_test10.csv` (from the `scripts/` folder in the repo), or paste the CSV path below.

The test set covers the full formality spectrum:
- **High formality / evening:** Aventus, No. 5, Tobacco Vanille
- **Medium / versatile:** Sauvage EDP, Jazz Club, Black Opium, Terre d'Hermès, Gypsy Water
- **Low / casual / summer:** Acqua di Giò, Wood Sage & Sea Salt

In [ ]:
import pandas as pd
from io import StringIO

# ── Option A: Upload vibescent_test10.csv ──────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(next(iter(uploaded)))

# ── Option B: Inline data (no upload needed) ───────────────────────────────
CSV_DATA = '''fragrance_id,name,brand,formality_score,fresh_warm_score,day_night_score,retrieval_text
test_001,Aventus,Creed,0.82,0.50,0.65,"Brand: Creed | Name: Aventus | Accords: Fruity, Woody, Smoky, Citrus | Top: Bergamot, Blackcurrant, Apple, Pineapple | Heart: Birch, Patchouli, Jasmine, Rose | Base: Musk, Oak Moss, Ambergris, Vanilla | Season: fall | Best for: formal event | Formality: high | Character: bold, sophisticated, powerful, charismatic, timeless | Vibe: A commanding signature for sharp tailoring and high-stakes occasions."
test_002,Sauvage EDP,Dior,0.45,0.30,0.40,"Brand: Dior | Name: Sauvage EDP | Accords: Woody, Spicy, Aromatic, Lavender | Top: Bergamot, Pepper | Heart: Lavender, Sichuan Pepper, Pink Pepper, Geranium | Base: Ambroxan, Cedar, Sandalwood | Season: all-season | Best for: casual outing | Formality: medium | Character: fresh, rugged, versatile, magnetic, effortless | Vibe: Clean and modern — works from a morning coffee run to a relaxed dinner."
test_003,No. 5 EDP,Chanel,0.90,0.60,0.70,"Brand: Chanel | Name: No. 5 EDP | Accords: Floral, Powdery, Aldehydic, Woody | Top: Aldehydes, Bergamot, Lemon, Neroli | Heart: Rose, Jasmine, Ylang-Ylang, Iris, Lily of the Valley | Base: Sandalwood, Vetiver, Musk, Ambergris | Season: spring | Best for: formal event | Formality: high | Character: iconic, elegant, classic, timeless, sophisticated | Vibe: The definitive formal fragrance — a black-tie gown in scent form."
test_004,Tobacco Vanille,Tom Ford,0.75,0.92,0.85,"Brand: Tom Ford | Name: Tobacco Vanille | Accords: Gourmand, Spicy, Woody, Sweet, Warm | Top: Tobacco Leaf, Spices | Heart: Tobacco Blossom, Jasmine, Vanilla, Ginger | Base: Sandalwood, Cocoa, Dried Fruits, Tonka Bean | Season: winter | Best for: evening party | Formality: high | Character: opulent, warm, sensuous, cozy, luxurious | Vibe: Rich winter evening warmth — pairs with velvet, heavy coats, and cashmere."
test_005,Acqua di Gio Profumo,Giorgio Armani,0.22,0.15,0.30,"Brand: Giorgio Armani | Name: Acqua di Gio Profumo | Accords: Aquatic, Woody, Marine, Smoky, Fresh | Top: Marine, Bergamot, Sage | Heart: Geranium, Rosemary, Incense | Base: Patchouli, Oakmoss, Musk | Season: summer | Best for: casual outing | Formality: low | Character: fresh, aquatic, breezy, clean, outdoorsy | Vibe: Salt air and open water — for relaxed summer days and anything involving light linen."
test_006,Wood Sage & Sea Salt,Jo Malone,0.18,0.10,0.20,"Brand: Jo Malone | Name: Wood Sage & Sea Salt | Accords: Aquatic, Aromatic, Woody, Fresh, Mineral | Top: Ambrette Seeds, Sea Notes, Citrus | Heart: Sage | Base: Driftwood, Mineral, Musk | Season: summer | Best for: beach day | Formality: low | Character: airy, natural, minimalist, coastal, effortless | Vibe: Bare skin and driftwood — for summer parties and weekends where the mood is light."
test_007,Replica Jazz Club,Maison Margiela,0.50,0.72,0.80,"Brand: Maison Margiela | Name: Replica Jazz Club | Accords: Woody, Spicy, Warm, Vanilla, Aromatic | Top: Pink Pepper, Lemon, Neroli | Heart: Rum, Clary Sage, Geranium | Base: Tobacco, Vanilla, Musk, Amber | Season: fall | Best for: evening party | Formality: medium | Character: warm, smoky, cozy, creative, nocturnal | Vibe: Dimly lit bar and improvised sets — ideal for creative industry nights out."
test_008,Black Opium,Yves Saint Laurent,0.55,0.75,0.78,"Brand: Yves Saint Laurent | Name: Black Opium | Accords: Gourmand, Floral, Sweet, Warm, Spicy | Top: Pink Pepper, Orange Blossom, Pear | Heart: Coffee, Jasmine, Bitter Almond | Base: White Musk, Patchouli, Vanilla, Cedar | Season: winter | Best for: night club | Formality: medium | Character: edgy, sweet, electrifying, dark, urban | Vibe: Late-night energy for streetwear nights — dark denim and statement accessories."
test_009,Terre d Hermes EDP,Hermes,0.52,0.55,0.45,"Brand: Hermes | Name: Terre d Hermes EDP | Accords: Woody, Earthy, Mineral, Citrus, Spicy | Top: Orange, Grapefruit, Flint | Heart: Pepper, Geranium | Base: Vetiver, Benzoin, Patchouli, Sandalwood | Season: fall | Best for: office | Formality: medium | Character: earthy, grounded, mineral, understated, professional | Vibe: Clean confidence for smart-casual and creative office environments."
test_010,Gypsy Water,Byredo,0.48,0.35,0.42,"Brand: Byredo | Name: Gypsy Water | Accords: Woody, Fresh, Citrus, Powdery, Incense | Top: Bergamot, Lemon, Pepper | Heart: Incense, Pine Needles, Juniper Berries | Base: Vanilla, Orris, Sandalwood | Season: spring | Best for: gallery opening | Formality: medium | Character: nomadic, artistic, free-spirited, ethereal, modern | Vibe: Worn art — for gallery openings and editorial settings calling for quiet originality."
'''

df = pd.read_csv(StringIO(CSV_DATA))
print(f'Loaded {len(df)} test fragrances')
display(df[['fragrance_id', 'name', 'brand', 'formality_score', 'fresh_warm_score', 'day_night_score']])


## Step 5 — Embed fragrance documents

In [ ]:
import numpy as np
from pathlib import Path

BATCH_SIZE = 8  # safe for A100 at float16

texts = df['retrieval_text'].tolist()
print(f'Embedding {len(texts)} documents...')

all_embs = []
for i in range(0, len(texts), BATCH_SIZE):
    batch_items = [{'text': t} for t in texts[i:i + BATCH_SIZE]]
    with torch.no_grad():
        result = embedder.process(batch_items)
    all_embs.append(result.cpu().float().numpy())
    print(f'  {min(i + BATCH_SIZE, len(texts))}/{len(texts)} done')

doc_embs = np.vstack(all_embs)
print(f'Document embeddings shape: {doc_embs.shape}')  # (10, 4096)

np.save('doc_embeddings_test10.npy', doc_embs)
print('Saved doc_embeddings_test10.npy')


## Step 6 — Text-only retrieval

Encodes occasion text as a query and ranks the 10 fragrances by cosine similarity.
No image involved — this is the text-only baseline.

In [ ]:
def cosine_retrieve(query_emb, doc_embs, df, top_k=5, label=''):
    """Rank fragrances by cosine similarity to a query embedding."""
    q = query_emb.flatten()
    q = q / (np.linalg.norm(q) + 1e-9)
    d = doc_embs / (np.linalg.norm(doc_embs, axis=1, keepdims=True) + 1e-9)
    scores = d @ q
    idx = np.argsort(scores)[::-1][:top_k]
    rows = []
    for rank, i in enumerate(idx, 1):
        row = df.iloc[i]
        rows.append({
            'rank': rank,
            'score': round(float(scores[i]), 4),
            'name': row['name'],
            'brand': row['brand'],
            'formality': row['formality_score'],
        })
    result = pd.DataFrame(rows)
    if label:
        result.insert(0, 'query', label)
    return result


# 4 test occasions from examples/occasions.json
TEST_OCCASIONS = [
    ('casual_day',        'Easy daytime coffee run in sneakers, relaxed denim, and a white tee.'),
    ('black_tie',         'Black tie gala with a tuxedo, crisp shirt, and formal evening presence.'),
    ('summer_party',      'Warm-weather rooftop party with breathable fabrics, clean colors, and bright energy.'),
    ('streetwear_night',  'Night-out streetwear with a hoodie, wide pants, sneakers, and a sharper urban edge.'),
]

text_results = {}
for occ_id, occ_text in TEST_OCCASIONS:
    print(f'\n{"="*60}')
    print(f'TEXT-ONLY  [{occ_id}]')
    print(f'Query: {occ_text[:80]}')
    print('='*60)
    with torch.no_grad():
        q_emb = embedder.process([{'text': occ_text}]).cpu().float().numpy()
    result = cosine_retrieve(q_emb, doc_embs, df)
    text_results[occ_id] = result.to_dict('records')
    display(result)


## Step 7 — Multimodal retrieval (text + outfit image)

Upload an outfit image, then run the same 4 occasions with the image included in the query.
The delta vs text-only shows the value of the visual signal.

**Good test images to try:**
- Tuxedo / black tie → should boost Aventus, No. 5, Tobacco Vanille
- White sneakers + denim → should boost Sauvage, Acqua di Giò, Wood Sage
- Dark streetwear → should boost Black Opium, Jazz Club

In [ ]:
# Upload an outfit image
from google.colab import files as colab_files
from PIL import Image
import IPython.display as ipd

print('Upload one outfit image (JPG or PNG):')
uploaded = colab_files.upload()

IMAGE_PATH = list(uploaded.keys())[0]
img = Image.open(IMAGE_PATH)
print(f'Loaded: {IMAGE_PATH}  {img.size}  {img.mode}')
ipd.display(img.resize((300, int(300 * img.height / img.width))))

# ── Alternative: download from URL ──────────────────────────────────────────
# import urllib.request
# urllib.request.urlretrieve('https://your-image-url.jpg', 'outfit.jpg')
# IMAGE_PATH = 'outfit.jpg'

# ── Alternative: mount Google Drive ─────────────────────────────────────────
# from google.colab import drive; drive.mount('/content/drive')
# IMAGE_PATH = '/content/drive/MyDrive/outfit.jpg'


In [ ]:
import json

mm_results = {}
for occ_id, occ_text in TEST_OCCASIONS:
    print(f'\n{"="*60}')
    print(f'MULTIMODAL  [{occ_id}]  + {IMAGE_PATH}')
    print('='*60)

    with torch.no_grad():
        q_emb_mm = embedder.process(
            [{'text': occ_text, 'image': IMAGE_PATH}]
        ).cpu().float().numpy()

    result = cosine_retrieve(q_emb_mm, doc_embs, df)
    mm_results[occ_id] = result.to_dict('records')
    display(result)

    # Show delta vs text-only
    text_top = [r['name'] for r in text_results[occ_id]]
    mm_top   = [r['name'] for r in mm_results[occ_id]]
    changed  = [n for n in mm_top if n not in text_top]
    if changed:
        print(f'  Image added to top-5: {changed}')
    else:
        print('  Top-5 unchanged from text-only.')


## Step 8 — Run on multiple images (ablation)

To generate the multimodal ablation table for the Week 2 report, run this block once per outfit image.
Each run appends results to `all_runs` keyed by image filename.

In [ ]:
# Accumulate results across multiple images
# Re-run this cell after each new IMAGE_PATH upload

if 'all_runs' not in dir():
    all_runs = {}

run_key = IMAGE_PATH
all_runs[run_key] = {}

for occ_id, occ_text in TEST_OCCASIONS:
    with torch.no_grad():
        q_emb_mm = embedder.process(
            [{'text': occ_text, 'image': IMAGE_PATH}]
        ).cpu().float().numpy()
    result = cosine_retrieve(q_emb_mm, doc_embs, df, top_k=10)
    all_runs[run_key][occ_id] = result.to_dict('records')

print(f'Stored run for {run_key}. Total runs: {len(all_runs)}')
print('Images covered:', list(all_runs.keys()))


## Step 9 — Save artifacts for repo integration

In [ ]:
import json, os
from google.colab import files as colab_files

# Save embeddings
np.save('doc_embeddings_test10.npy', doc_embs)

# Save text-only results
with open('multimodal_text_only_results.json', 'w') as f:
    json.dump(text_results, f, indent=2)

# Save multimodal results (all images)
with open('multimodal_mm_results.json', 'w') as f:
    json.dump(all_runs if 'all_runs' in dir() else mm_results, f, indent=2)

# Summary: top-1 per occasion per mode
print('\n=== Top-1 summary ========================')
print(f'{"Occasion":<22} {"Text-only top-1":<25} {"Multimodal top-1"}')
print('-' * 70)
for occ_id, _ in TEST_OCCASIONS:
    t1 = text_results[occ_id][0]
    m1 = mm_results.get(occ_id, [{}])[0]
    t_str = f"{t1['brand']} {t1['name']} ({t1['score']:.3f})"
    m_str = f"{m1.get('brand','')} {m1.get('name','')} ({m1.get('score', 0):.3f})" if m1 else 'n/a'
    print(f'{occ_id:<22} {t_str:<38} {m_str}')

print('\nFiles to download and commit to artifacts/:')
for fname in ['doc_embeddings_test10.npy', 'multimodal_text_only_results.json', 'multimodal_mm_results.json']:
    print(f'  {fname}  ({os.path.getsize(fname)} bytes)')

# Download all to local machine
colab_files.download('doc_embeddings_test10.npy')
colab_files.download('multimodal_text_only_results.json')
colab_files.download('multimodal_mm_results.json')


## Checklist before closing

After running this notebook, place the downloaded files in the repo:

```
artifacts/
  multimodal_test10/
    doc_embeddings_test10.npy        ← document embedding matrix (10, 4096)
    multimodal_text_only_results.json
    multimodal_mm_results.json       ← one entry per outfit image
```

Then update `results/week2_report.md` Section 4 (Multimodal Ablation) with:
- The top-1 summary table above
- Screenshot of at least one interesting image→fragrance delta
- Note which occasions changed top-5 when image was added vs text-only

**For integration with the full 500-row corpus:** replace `vibescent_test10.csv` with `data/vibescent_enriched.csv` in Step 4 and re-run. Pre-compute and save `doc_embeddings_enriched.npy` — the model cannot run live during the demo.
